# Finalny pipeline — wytworzenie splitów

Jeden notebook end-to-end: od surowych danych do gotowych splitów treningowych.

**Wejście:** `data/pjm_data.csv` (metadane migających) + `data/texts_eng.h5` (tłumaczenia + flagi źródła)

**Wyjście:**
- `data/processed_texts.csv` — teksty po normalizacji/NLP + metryki jakości
- `data/split_{train,val,test}.csv` — split speaker-stratified (zero leakage)
- `data/split_{train,val,test}_filtered.csv` — po filtrze jakości
- `data/split_{train,val,test}_ms.csv` — wariant multisigner (jak PHOENIX-2014T)

Etapy: **1** load+enrich → **2** normalizacja+NLP → **3** maskowanie+metryki → **4** selekcja (keep/drop) →
**5** split speaker-stratified → **6** filtr jakości → **7** multisigner → **8** weryfikacja liczności.

> Cel treningowy = oryginalna kolumna `text`. Kolumny `masked`/`mask_ratio` służą wyłącznie filtracji.


## Konfiguracja


In [1]:
import random
import unicodedata
import json as _json
from collections import Counter, defaultdict

import contractions
import pandas as pd
import spacy
from tqdm import tqdm
from transformers import AutoTokenizer

from hdf5_manager import HDF5Loader

tqdm.pandas()

DATA = "./data"
H5_PATH = f"{DATA}/texts_eng.h5"
PJM_PATH = f"{DATA}/pjm_data.csv"

# --- progi pipeline (jedyne miejsce do strojenia) ---
RARE_THRESHOLD = 3            # słowo "rzadkie": <= tylu wystąpień w korpusie
MASK_RATIO_THRESHOLD = 0.3    # selekcja: drop jeśli >30% tokenów zamaskowanych
T5_MAX_SUBTOKENS = 60         # okno dekodera flan-t5-base (max_txt_len=64)
MIN_WORDS = 3                 # filtr jakości: minimum słów
MIN_VOCAB_PCT = 50.0          # filtr jakości: najrzadsze słowo w top X% korpusu
SPLIT_SEED = 42               # powtarzalność shuffle (multisigner)
MS_TRAIN_RATIO, MS_VAL_RATIO = 0.80, 0.10


## Etap 1 — Wczytanie i wzbogacenie danych

`texts_eng.h5` → tekst + flaga źródła; `pjm_data.csv` → metadane migacza (po `video_id` + `speaker_side`).


In [2]:
hdf5 = HDF5Loader(H5_PATH)
data = hdf5.load()
hdf5_flags = hdf5.load_flags()
hdf5.close()

df = pd.DataFrame({"key": list(data.keys()), "text": list(data.values())})
df["word_count"] = df["text"].str.split().str.len()
df["source_flag"] = df["key"].map(hdf5_flags)

df_meta = pd.read_csv(PJM_PATH)
df["video_id"] = df["key"].str.split("_").str[0].astype(int)
_side_map = {"L": "left", "R": "right", "M": "-"}
df["speaker_side"] = df["key"].str.split("_").str[2].map(_side_map)

n_before = len(df)
df = df.merge(
    df_meta[["video_id", "speaker_side", "speaker_id", "sex", "age", "task_id", "video_type", "special_case"]],
    on=["video_id", "speaker_side"],
    how="left",
)
assert len(df) == n_before, f"Merge zmienił liczbę wierszy: {n_before} -> {len(df)}"
df["interaction_type"] = df["video_type"].str.split(";").str[0]

print(f"Wczytano {len(df)} nagrań, {df['speaker_id'].nunique()} migaczy")
print(f"Bez speaker_id: {df['speaker_id'].isna().sum()}")


Wczytano 39817 nagrań, 151 migaczy
Bez speaker_id: 0


## Etap 2 — Normalizacja tekstu i NLP

Lowercase → rozwinięcie skrótów → usunięcie diakrytyków → usunięcie interpunkcji. Następnie POS/NER (spaCy) i lematyzacja.


In [5]:
def to_ascii(text):
    # "café" -> "cafe", "Przemyśl" -> "Przemysl"
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")

df["clean"] = df["text"].str.lower()
df["clean"] = df["clean"].apply(contractions.fix)
df["clean"] = df["clean"].apply(to_ascii)
df["clean"] = df["clean"].str.replace(r"[^a-z0-9\s]", "", regex=True)
df["clean_word_count"] = df["clean"].str.split().str.len()

nlp = spacy.load("en_core_web_sm")

def analyze_nlp(text):
    doc = nlp(text)
    return {
        "pos": [(token.lemma_.lower(), token.pos_) for token in doc],
        "ents": [(ent.text, ent.label_) for ent in doc.ents],
    }

df["nlp"] = df["clean"].progress_apply(analyze_nlp)

def lemmatize(text):
    return " ".join(token.lemma_ for token in nlp(text))

df["lemmatized"] = df["clean"].progress_apply(lemmatize)


100%|██████████| 39817/39817 [03:03<00:00, 217.39it/s]


## Etap 3 — Maskowanie rzadkich słów + metryki jakości

Rzadkie słowa zastępowane tagiem `<NER>`/`<POS>`/`<UNK>`. Liczymy `mask_ratio`, `final_wc` oraz `min_vocab_pct`
(pokrycie korpusu przez najrzadsze słowo w zdaniu). Zapis kanonicznego `processed_texts.csv`.


In [6]:
word_freq = Counter()
for text in df["lemmatized"]:
    word_freq.update(str(text).split())

rare_words = {w for w, c in word_freq.items() if c <= RARE_THRESHOLD}
print(f"Rzadkie słowa (<= {RARE_THRESHOLD}): {len(rare_words)} / {len(word_freq)}")

def mask_rare(row):
    nlp_data = row["nlp"]
    ner_map = {}
    for ent_text, ent_label in nlp_data["ents"]:
        for w in ent_text.lower().split():
            ner_map[w] = ent_label
    pos_map = {w.lower(): pos for w, pos in nlp_data["pos"]}

    result = []
    for w in row["lemmatized"].split():
        if w not in rare_words:
            result.append(w)
        elif w in ner_map:
            result.append(f"<{ner_map[w]}>")
        elif w in pos_map:
            result.append(f"<{pos_map[w]}>")
        else:
            result.append("<UNK>")
    return " ".join(result)

df["masked"] = df.apply(mask_rare, axis=1)
df["mask_ratio"] = df["masked"].apply(lambda x: x.count("<") / max(len(x.split()), 1))
df["final_wc"] = df["masked"].str.split().str.len()

# min_vocab_pct: skumulowane pokrycie korpusu przez najrzadsze słowo zdania
total_occurrences = sum(word_freq.values())
cumsum = 0.0
word_vocab_pct = {}
for word, count in word_freq.most_common():
    cumsum += count
    word_vocab_pct[word] = round(cumsum / total_occurrences * 100, 2)

df["min_vocab_pct"] = df["lemmatized"].apply(
    lambda t: max((word_vocab_pct.get(w, 100.0) for w in str(t).split()), default=0.0)
)

# Zapis kanonicznego artefaktu (nlp serializowany jako JSON)
_to_save = df.copy()
_to_save["nlp"] = _to_save["nlp"].apply(_json.dumps)
_to_save.to_csv(f"{DATA}/processed_texts.csv", index=False)
print(f"Zapisano {len(_to_save)} wierszy -> processed_texts.csv")


Rzadkie słowa (<= 3): 2540 / 4660
Zapisano 39817 wierszy -> processed_texts.csv


## Etap 4 — Selekcja (keep / drop)

Pięć sekwencyjnych filtrów. Pierwszy pasujący filtr decyduje o odrzuceniu.
Flagi informacyjne (`has_rare_ner`/`short`/`one_hand`) z `selection.ipynb` pomijamy — nie wpływają na splity.


In [7]:
df["decision"] = "keep"
df["reason"] = "keep"
df["flag"] = ""

filter_log = []

def apply_filter(name, mask):
    active = (df["decision"] == "keep") & mask
    df.loc[active, "decision"] = "drop"
    df.loc[active, "reason"] = name
    filter_log.append((name, int(active.sum()), int((df["decision"] == "keep").sum())))

apply_filter("invalid_source", df["source_flag"] == "invalid")
apply_filter("empty", (df["final_wc"] == 0) | df["clean"].str.fullmatch(r"[\d\s]*", na=True))
apply_filter("single_word", df["word_count"] == 1)
apply_filter("high_mask_ratio", df["mask_ratio"] > MASK_RATIO_THRESHOLD)

t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
df["subtoken_len"] = df["text"].apply(lambda x: len(t5_tokenizer.encode(str(x))))
apply_filter("exceeds_decoder_window", df["subtoken_len"] > T5_MAX_SUBTOKENS)

print(f"{'Filtr':<28}{'Drop':>8}{'Remaining':>12}")
print("-" * 48)
for name, dropped, remaining in filter_log:
    print(f"{name:<28}{dropped:>8}{remaining:>12}")
kept_total = int((df["decision"] == "keep").sum())
print("-" * 48)
print(f"{'KEEP':<28}{'':>8}{kept_total:>12}  ({kept_total/len(df)*100:.1f}%)")


Filtr                           Drop   Remaining
------------------------------------------------
invalid_source                    10       39807
empty                              2       39805
single_word                     5527       34278
high_mask_ratio                  362       33916
exceeds_decoder_window            19       33897
------------------------------------------------
KEEP                                       33897  (85.1%)


## Etap 5 — Split speaker-stratified (zero leakage)

Migacze sortowani malejąco wg liczby segmentów i przydzielani round-robin do train/val/test —
żaden migacz nie pojawia się w dwóch zbiorach.


In [8]:
SPLIT_COLS = ["key", "speaker_id", "interaction_type", "sex", "age",
              "min_vocab_pct", "decision", "reason", "flag"]

kept = df[df["decision"] == "keep"].copy()
speaker_counts = kept["speaker_id"].value_counts().sort_values(ascending=False)
speaker_sorted = speaker_counts.index.tolist()

assignments = {spk: ["train", "val", "test"][i % 3] for i, spk in enumerate(speaker_sorted)}
spk_of = {name: [s for s, a in assignments.items() if a == name] for name in ("train", "val", "test")}

split_dfs = {}
for name in ("train", "val", "test"):
    sub = kept[kept["speaker_id"].isin(spk_of[name])][SPLIT_COLS].copy()
    sub.to_csv(f"{DATA}/split_{name}.csv", index=False)
    split_dfs[name] = sub

# zero leakage + brak zgubionych segmentów
assert not (set(spk_of["train"]) & set(spk_of["val"])), "Leakage train/val"
assert not (set(spk_of["train"]) & set(spk_of["test"])), "Leakage train/test"
assert not (set(spk_of["val"]) & set(spk_of["test"])), "Leakage val/test"
split_total = sum(len(v) for v in split_dfs.values())
assert split_total == len(kept), f"Zgubiono segmenty: {len(kept)} -> {split_total}"

for name, sub in split_dfs.items():
    print(f"{name:<6} {len(sub):>7} segmentów, {sub['speaker_id'].nunique():>3} migaczy")
print(f"RAZEM  {split_total:>7} (== keep: {len(kept)})")


train    11448 segmentów,  51 migaczy
val      11293 segmentów,  50 migaczy
test     11156 segmentów,  50 migaczy
RAZEM    33897 (== keep: 33897)


## Etap 6 — Filtr jakości → splity `_filtered`

`word_count >= 3` & `min_vocab_pct >= 50` & `source_flag != invalid`. Metryki dołączane z głównej ramki po `key`.


In [9]:
quality = df[["key", "word_count", "source_flag"]]

def filter_split(d):
    mask = (
        (d["word_count"] >= MIN_WORDS)
        & (d["min_vocab_pct"] >= MIN_VOCAB_PCT)
        & (d["source_flag"] != "invalid")
    )
    return d[mask]

filtered_dfs = {}
print(f"{'Split':<6}{'Oryg.':>9}{'Po filtrze':>12}{'Usunięto':>10}")
print("-" * 37)
for name in ("train", "val", "test"):
    orig = split_dfs[name]
    merged = orig.merge(quality, on="key", how="left")
    keep_keys = set(filter_split(merged)["key"])
    out = orig[orig["key"].isin(keep_keys)].copy()
    out.to_csv(f"{DATA}/split_{name}_filtered.csv", index=False)
    filtered_dfs[name] = out
    print(f"{name:<6}{len(orig):>9}{len(out):>12}{len(orig)-len(out):>10}")

filtered_total = sum(len(v) for v in filtered_dfs.values())
print("-" * 37)
print(f"{'RAZEM':<6}{split_total:>9}{filtered_total:>12}{split_total-filtered_total:>10}")


Split     Oryg.  Po filtrze  Usunięto
-------------------------------------
train     11448       10063      1385
val       11293        9958      1335
test      11156        9911      1245
-------------------------------------
RAZEM     33897       29932      3965


## Etap 7 — Wariant multisigner

Każdy migacz proporcjonalnie (80/10/10) w każdym zbiorze — porównywalne z PHOENIX-2014T multisigner.
Wejście: przefiltrowane splity z Etapu 6.


In [10]:
random.seed(SPLIT_SEED)

rows = []
for name in ("train", "val", "test"):
    rows.extend(filtered_dfs[name].to_dict("records"))

by_signer = defaultdict(list)
for row in rows:
    by_signer[row["speaker_id"]].append(row)

ms = {"train": [], "val": [], "test": []}
for signer_id, samples in by_signer.items():
    random.shuffle(samples)
    n = len(samples)
    n_train = max(1, round(n * MS_TRAIN_RATIO))
    n_val = max(1, round(n * MS_VAL_RATIO))
    n_test = max(1, n - n_train - n_val)
    while n_train + n_val + n_test > n:
        if n_test > 1: n_test -= 1
        elif n_val > 1: n_val -= 1
        else: n_train -= 1
    ms["train"].extend(samples[:n_train])
    ms["val"].extend(samples[n_train:n_train + n_val])
    ms["test"].extend(samples[n_train + n_val:n_train + n_val + n_test])

ms_cols = list(filtered_dfs["train"].columns)
for name in ("train", "val", "test"):
    pd.DataFrame(ms[name], columns=ms_cols).to_csv(f"{DATA}/split_{name}_ms.csv", index=False)
    print(f"{name:<6} {len(ms[name]):>7} segmentów")

ms_total = sum(len(v) for v in ms.values())
all_signers = set(by_signer)
for name in ("train", "val", "test"):
    present = {r["speaker_id"] for r in ms[name]}
    assert present == all_signers, f"Brakuje migaczy w MS/{name}"
print(f"OK — wszyscy {len(all_signers)} migaczy w każdym splicie MS")


train    23943 segmentów
val       3000 segmentów
test      2989 segmentów
OK — wszyscy 151 migaczy w każdym splicie MS


## Etap 8 — Weryfikacja liczności (czy zgadza się na końcu)


In [11]:
print("=== Bilans liczności przez cały pipeline ===")
print(f"Surowe nagrania           : {len(df):>7}")
print(f"Po selekcji (keep)        : {len(kept):>7}")
print(f"Suma splitów SI           : {split_total:>7}   (== keep: {split_total == len(kept)})")
print(f"Suma splitów _filtered    : {filtered_total:>7}")
print(f"Suma splitów _ms          : {ms_total:>7}   (== filtered: {ms_total == filtered_total})")
print()

# Twarde asercje — nic nie gubimy ani nie duplikujemy
assert split_total == len(kept), "SI: suma splitów != keep"
assert filtered_total == len(filter_split(
    pd.concat(split_dfs.values()).merge(quality, on="key", how="left")
)), "Filtered: suma != całość po filtrze"
assert ms_total == filtered_total, "MS: suma != filtered (multisigner tylko redystrybuuje)"

# Zero leakage w SI (po kluczach)
keys = {n: set(filtered_dfs[n]["key"]) for n in ("train", "val", "test")}
assert not (keys["train"] & keys["val"]), "Leakage train/val"
assert not (keys["train"] & keys["test"]), "Leakage train/test"
assert not (keys["val"] & keys["test"]), "Leakage val/test"

print("WSZYSTKIE ASERCJE OK — liczności spójne, zero leakage.")


=== Bilans liczności przez cały pipeline ===
Surowe nagrania           :   39817
Po selekcji (keep)        :   33897
Suma splitów SI           :   33897   (== keep: True)
Suma splitów _filtered    :   29932
Suma splitów _ms          :   29932   (== filtered: True)

WSZYSTKIE ASERCJE OK — liczności spójne, zero leakage.
